<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Gemini API and VertexAI</h1>
<h1>Fundamentals</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from pprint import pprint
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold
import vertexai
from vertexai.generative_models import GenerativeModel, Part, GenerationConfig

import watermark
%load_ext watermark
%matplotlib inline


We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.10.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 89a61e40d04ebc1e5512ea5b07fd4330e8b29ff1

matplotlib: 3.10.8
numpy     : 2.4.2
pandas    : 3.0.1
vertexai  : 1.71.1
watermark : 2.6.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# 1. Introduction to the Gemini API

## The Gemini Model Family

Google's **Gemini** is a family of natively multimodal foundation models capable of processing text, images, audio, video, and code.

In [4]:
# NOTE: Set your API key before running this cell:
#   export GEMINI_API_KEY='your-key-here'
# Or set it programmatically (not recommended for shared notebooks):
#   os.environ["GEMINI_API_KEY"] = "your-key-here"

api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY environment variable not set. "
        "Please set it before running this notebook."
    )

genai.configure(api_key=api_key)
print("Gemini API configured successfully.")

Gemini API configured successfully.


In [5]:
# List all available Gemini models
print("Available models:")
print("-" * 60)

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(f"  Name       : {model.name}")
        print(f"  Display    : {model.display_name}")
        print(f"  Description: {model.description[:80]}..." if len(model.description) > 80 else f"  Description: {model.description}")
        print()

Available models:
------------------------------------------------------------
  Name       : models/gemini-2.5-flash
  Display    : Gemini 2.5 Flash
  Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports ...

  Name       : models/gemini-2.5-pro
  Display    : Gemini 2.5 Pro
  Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro

  Name       : models/gemini-2.0-flash
  Display    : Gemini 2.0 Flash
  Description: Gemini 2.0 Flash

  Name       : models/gemini-2.0-flash-001
  Display    : Gemini 2.0 Flash 001
  Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for ...

  Name       : models/gemini-2.0-flash-lite-001
  Display    : Gemini 2.0 Flash-Lite 001
  Description: Stable version of Gemini 2.0 Flash-Lite

  Name       : models/gemini-2.0-flash-lite
  Display    : Gemini 2.0 Flash-Lite
  Description: Gemini 2.0 Flash-Lite

  Name       : models/gemini-2.5-flash-preview-tts
  Display    : Gem

Inspect a specific model's properties

In [6]:
model_info = genai.get_model("models/gemini-2.0-flash")

print(f"Model Name          : {model_info.name}")
print(f"Display Name        : {model_info.display_name}")
print(f"Input Token Limit   : {model_info.input_token_limit:,}")
print(f"Output Token Limit  : {model_info.output_token_limit:,}")
print(f"Supported Methods   : {model_info.supported_generation_methods}")
print(f"Temperature Range   : {model_info.temperature} (default)")
print(f"TopP                : {model_info.top_p}")
print(f"TopK                : {model_info.top_k}")

Model Name          : models/gemini-2.0-flash
Display Name        : Gemini 2.0 Flash
Input Token Limit   : 1,048,576
Output Token Limit  : 8,192
Supported Methods   : ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Temperature Range   : 1.0 (default)
TopP                : 0.95
TopK                : 40


## First API Call: Simple Text Generation

Create a model instance and generate a simple response

In [7]:
model = genai.GenerativeModel("gemini-2.5-flash")

prompt = "Explain the concept of attention mechanisms in neural networks in 3 sentences."

response = model.generate_content(prompt)

print("Prompt:")
print(f"  {prompt}")
print()
print("Response:")
print(response.text)

Prompt:
  Explain the concept of attention mechanisms in neural networks in 3 sentences.

Response:
Attention mechanisms enable neural networks to dynamically weigh the importance of different parts of their input data when making predictions. Instead of processing all information uniformly, the network learns to focus on the most relevant features or sequence elements, much like how humans selectively concentrate. This allows models to handle long sequences effectively, improve performance, and provide insights into what information was crucial for a given output.


## Generation Configuration

Using `GenerationConfig` to control sampling behavior

In [8]:
generation_config = genai.GenerationConfig(
    temperature=0.2,          # Low temperature for factual, consistent output
    top_p=0.9,                # Nucleus sampling
    top_k=40,                 # Restrict to top 40 tokens
    max_output_tokens=512,    # Limit response length
    stop_sequences=["##"],    # Stop if the model produces '##'
)

model_configured = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=generation_config,
)

response = model_configured.generate_content(
    "List the top 5 use cases for large language models in enterprise settings. "
    "Format as a numbered list with a one-sentence explanation for each."
)

print(response.text)
print()
print("--- Usage Metadata ---")
print(f"Prompt tokens    : {response.usage_metadata.prompt_token_count}")
print(f"Candidates tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens     : {response.usage_metadata.total_token_count}")

Here are the top 5 use cases for large language models in enterprise settings:

1.

--- Usage Metadata ---
Prompt tokens    : 30
Candidates tokens: 19
Total tokens     : 538


## Safety Settings

Gemini includes built-in safety filters that can be adjusted per harm category:

| Category | Description |
|---|---|
| `HARM_CATEGORY_HARASSMENT` | Bullying, threatening, or abusive content |
| `HARM_CATEGORY_HATE_SPEECH` | Content targeting identity characteristics |
| `HARM_CATEGORY_SEXUALLY_EXPLICIT` | Explicit sexual content |
| `HARM_CATEGORY_DANGEROUS_CONTENT` | Content promoting dangerous activities |

**Block thresholds** (from most to least restrictive):
`BLOCK_LOW_AND_ABOVE` > `BLOCK_MEDIUM_AND_ABOVE` > `BLOCK_ONLY_HIGH` > `BLOCK_NONE`

In [9]:
safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT:        HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH:       HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
}

model_safe = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    safety_settings=safety_settings,
)

response = model_safe.generate_content("What are the main risks of deploying AI in healthcare?")

# Check safety ratings
print("Safety Ratings:")
for rating in response.candidates[0].safety_ratings:
    print(f"  {rating.category.name}: {rating.probability.name}")
print()
print("Response:")
print(response.text)

Safety Ratings:

Response:
Deploying Artificial Intelligence (AI) in healthcare holds immense promise, but it also introduces several significant risks that need careful consideration and mitigation. These risks span technical, ethical, legal, and operational domains.

Here are the main risks:

1.  **Patient Safety and Clinical Errors:**
    *   **Misdiagnosis and Suboptimal Treatment:** AI models, if trained on biased or incomplete data, or if they encounter situations outside their training scope, can make incorrect predictions or recommendations, leading to misdiagnosis, delayed treatment, or inappropriate interventions.
    *   **Over-reliance by Clinicians:** Healthcare professionals might become overly dependent on AI suggestions, potentially reducing their critical thinking and ability to detect AI errors, leading to "automation bias."
    *   **Lack of Explainability (Black Box Problem):** Many advanced AI models (especially deep learning) are "black boxes," meaning it's diffic

## Multi-turn Conversations (Chat)

The Gemini SDK supports stateful multi-turn conversations via `start_chat()`.

In [10]:
model = genai.GenerativeModel("gemini-2.5-flash")
chat = model.start_chat(history=[])

turns = [
    "I'm learning about neural networks. Can you briefly explain what a neuron does?",
    "How does that relate to the transformer architecture?",
    "And what makes Gemini different from earlier transformers?",
]

for user_message in turns:
    print(f"USER: {user_message}")
    response = chat.send_message(user_message)
    print(f"GEMINI: {response.text}")
    print("-" * 60)

print(f"\nTotal turns in history: {len(chat.history)}")

USER: I'm learning about neural networks. Can you briefly explain what a neuron does?
GEMINI: Think of an artificial neuron as a very simplified model of a biological neuron. Here's what it fundamentally does:

1.  **Receives Inputs:** It takes in one or more numerical values (signals) from other neurons or raw data.
2.  **Applies Weights:** Each input signal is multiplied by a specific **weight**. Weights are learnable parameters that determine the "importance" of each input. A higher weight means that input has a stronger influence.
3.  **Sums Them Up (and adds Bias):** All the weighted inputs are then added together. A **bias** term (another learnable parameter) is also added to this sum. The bias allows the neuron to shift its activation threshold.
4.  **Applies an Activation Function:** The combined sum (weighted sum + bias) is then passed through a non-linear function called an **activation function**. This function decides whether the neuron "fires" (becomes active) and what the

# 3. Overview of Vertex AI

**Vertex AI** is Google Cloud's unified machine learning platform. It provides end-to-end ML capabilities from data preparation and model training to deployment, monitoring, and MLOps — all under one roof.

**Prerequisites:**
1. Google Cloud project with billing enabled
2. Enable the Vertex AI API: `gcloud services enable aiplatform.googleapis.com`
3. Authenticate: `gcloud auth application-default login`
4. Install the SDK: `pip install google-cloud-aiplatform`
5. Set environment variables:
   ```bash
   export GCP_PROJECT='your-project-id'
   export GCP_LOCATION='us-central1'
   ```

In [11]:
GCP_PROJECT  = os.environ.get("GCP_PROJECT")
GCP_LOCATION = os.environ.get("GCP_LOCATION", "us-central1")

if not GCP_PROJECT:
    raise EnvironmentError(
        "GCP_PROJECT environment variable not set. "
        "Please set it to your Google Cloud project ID."
    )

# Initialize the Vertex AI SDK
vertexai.init(project=GCP_PROJECT, location=GCP_LOCATION)

print(f"Vertex AI initialized.")
print(f"  Project  : {GCP_PROJECT}")
print(f"  Location : {GCP_LOCATION}")

Vertex AI initialized.
  Project  : gen-lang-client-0667691803
  Location : us-central1


In [12]:
# Using the Vertex AI GenerativeModel
vertex_model = GenerativeModel("gemini-2.5-flash")

vertex_response = vertex_model.generate_content(
    "What are the three most important considerations when deploying LLMs in production?"
)
print("Response via Vertex AI:")
print(vertex_response.text)

Response via Vertex AI:
Deploying LLMs in production involves navigating a complex landscape of technical, ethical, and operational challenges. While many factors are important, the three most critical considerations are:

1.  **Model Performance, Reliability, and Hallucination Mitigation:**
    *   **Accuracy & Relevance:** Does the LLM consistently provide correct, useful, and on-topic answers for your specific use case? This is paramount for user trust and utility.
    *   **Hallucination Mitigation:** LLMs are known to "hallucinate" or generate factually incorrect information while sounding confident. In production, this can lead to severe consequences. Strategies like Retrieval Augmented Generation (RAG), strict guardrails, fact-checking mechanisms, and human-in-the-loop validation are crucial.
    *   **Latency & Throughput:** How quickly can the model respond to requests, and how many requests can it handle simultaneously? User experience often hinges on low latency, while cost-

In [13]:
# Generation config with Vertex AI
vertex_config = GenerationConfig(
    temperature=0.3,
    top_p=0.95,
    top_k=40,
    max_output_tokens=1024,
)

vertex_model_configured = GenerativeModel(
    "gemini-2.5-flash",
    generation_config=vertex_config,
)

structured_prompt = """You are an expert ML engineer. 
Provide a concise checklist (5 items) for productionizing a generative AI application.
Format: numbered list, one line per item."""

vertex_response = vertex_model_configured.generate_content(structured_prompt)

print("Structured response via Vertex AI (with config):")
print(vertex_response.text)
print()
print("--- Usage Metadata ---")
print(vertex_response.usage_metadata)

Structured response via Vertex AI (with config):
Here's a concise checklist for productionizing a generative AI application:

1.  **Robust Deployment & Scalability:** Implement efficient inference serving, auto-scaling, and low-latency API endpoints.
2.  **Comprehensive Monitoring & Observability:** Track model quality (e.g., coherence, relevance), performance, cost, and user interaction metrics.
3.  **Safety & Responsible AI Guardrails:** Integrate content moderation, bias detection, and misuse prevention mechanisms (e.g., input/output filters).
4.  **Continuous Improvement & Feedback Loops:** Establish pipelines for data collection, human feedback, and iterative model fine-tuning/updates.
5.  **Cost Optimization & Resource Management:** Proactively manage inference costs, optimize hardware utilization, and explore model compression techniques.

--- Usage Metadata ---
prompt_token_count: 36
candidates_token_count: 162
total_token_count: 897



In [14]:
# System instructions: Vertex AI supports system-level persona
system_instruction_model = GenerativeModel(
    "gemini-2.5-flash",
    system_instruction="""You are a senior data scientist at a Fortune 500 company.
    You communicate in a concise, technical, and results-focused manner.
    Always back up your statements with data or industry best practices."""
)

response = system_instruction_model.generate_content(
    "What is the most effective way to evaluate an LLM for a customer support use case?"
)

print("Response with system instruction:")
print(response.text)

Response with system instruction:
The most effective way to evaluate an LLM for a customer support use case involves a multi-faceted approach, combining automated metrics, rigorous human evaluation, and ultimately, real-world business impact validation. This ensures alignment with both technical performance and strategic business objectives.

**1. Offline Automated Evaluation (Scalability & Efficiency):**
Leverage a comprehensive, representative dataset of historical customer queries and corresponding desired resolutions. This dataset should include common FAQs, complex scenarios, and edge cases.

*   **Factuality & Hallucination Rate:** Utilize metrics like RAGAS (specifically **Faithfulness** and **Answer Relevancy**). Faithfulness evaluates if the generated answer is grounded in the provided context, while Answer Relevancy measures how pertinent the answer is to the query. For knowledge-based systems, ground-truth fact-checking against the knowledge base is critical.
    *   *Indust

## Comparing API Calls: Direct API vs Vertex AI

Both paths use nearly identical Python interfaces — the main difference is initialization and authentication:

**Direct API (google-generativeai):**
```python
import google.generativeai as genai

genai.configure(api_key="YOUR_API_KEY")
model = genai.GenerativeModel("gemini-2.0-flash")
response = model.generate_content("Hello, Gemini!")
print(response.text)
```

**Vertex AI (google-cloud-aiplatform):**
```python
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project="your-project", location="us-central1")
model = GenerativeModel("gemini-2.0-flash")
response = model.generate_content("Hello, Gemini!")
print(response.text)
```

The core API surface (`generate_content`, `GenerationConfig`, `start_chat`) is essentially the same in both SDKs, making migration between the two straightforward.

In [15]:
# Side-by-side token counting: both APIs expose count_tokens

test_prompt = "Explain what Vertex AI is in one paragraph."

# Direct API
direct_model = genai.GenerativeModel("gemini-2.5-flash")
direct_count = direct_model.count_tokens(test_prompt)
print(f"Direct API - Token count for test prompt: {direct_count.total_tokens}")

# Vertex AI
vertex_count = vertex_model.count_tokens(test_prompt)
print(f"Vertex AI  - Token count for test prompt: {vertex_count.total_tokens}")

Direct API - Token count for test prompt: 9
Vertex AI  - Token count for test prompt: 9


<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>